# Phase 2 — Load the data and build the working sample

`US_Accidents_March23_sampled_500k.csv` is already a 500K-row sample of the 7.7M-row full dataset,
so no chunked sampling is needed. This notebook checks that the sample is representative
(Severity shares, years, states) and saves it as Parquet (smaller, typed, loads much faster than CSV).

In [1]:
from pathlib import Path
import pandas as pd

RAW = Path("../US_Accidents_March23_sampled_500k.csv")
DATA = Path("../data")
DATA.mkdir(exist_ok=True)

df = pd.read_csv(RAW)
df.shape

(500000, 46)

In [2]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 46 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ID                     500000 non-null  str    
 1   Source                 500000 non-null  str    
 2   Severity               500000 non-null  int64  
 3   Start_Time             500000 non-null  str    
 4   End_Time               500000 non-null  str    
 5   Start_Lat              500000 non-null  float64
 6   Start_Lng              500000 non-null  float64
 7   End_Lat                279623 non-null  float64
 8   End_Lng                279623 non-null  float64
 9   Distance(mi)           500000 non-null  float64
 10  Description            499999 non-null  str    
 11  Street                 499309 non-null  str    
 12  City                   499981 non-null  str    
 13  County                 500000 non-null  str    
 14  State                  500000 non-null  str    

## Severity: sample vs full dataset

In [3]:
# Published counts of the full release (US_Accidents_March23.csv, 7,728,394 rows)
FULL = pd.Series({1: 67_366, 2: 6_156_981, 3: 1_299_337, 4: 204_710})

sev = df["Severity"].value_counts().sort_index().to_frame("sample_count")
sev["sample_%"] = (sev["sample_count"] / len(df) * 100).round(2)
sev["full_count"] = FULL
sev["full_%"] = (FULL / FULL.sum() * 100).round(2)
sev["diff_pp"] = (sev["sample_%"] - sev["full_%"]).round(2)
sev

,sample_count,sample_%,full_count,full_%,diff_pp
Severity,,,,,
1,4274,0.85,67366,0.87,-0.02
2,398142,79.63,6156981,79.67,-0.04
3,84520,16.90,1299337,16.81,0.09
4,13064,2.61,204710,2.65,-0.04


## Rows per year (and Severity mix per year)

In [4]:
year = df["Start_Time"].str[:4].astype(int)  # first 4 chars = year; no datetime parsing needed here
by_year = pd.crosstab(year, df["Severity"], margins=True, margins_name="total")
by_year

Severity,1,2,3,4,total
Start_Time,,,,,
2016,19,17573,8190,881,26663
2017,17,29852,15093,1552,46514
2018,12,37091,18863,1612,57578
2019,16,44480,15518,1838,61852
2020,1808,60266,12061,2020,76155
2021,7,90103,9691,1939,101740
2022,2395,103487,5104,2748,113734
2023,0,15290,0,474,15764
total,4274,398142,84520,13064,500000


In [5]:
# Severity share (%) inside each year -> early sign of drift
pd.crosstab(year, df["Severity"], normalize="index").mul(100).round(1)

Severity,1,2,3,4
Start_Time,,,,
2016,0.1,65.9,30.7,3.3
2017,0.0,64.2,32.4,3.3
2018,0.0,64.4,32.8,2.8
2019,0.0,71.9,25.1,3.0
2020,2.4,79.1,15.8,2.7
2021,0.0,88.6,9.5,1.9
2022,2.1,91.0,4.5,2.4
2023,0.0,97.0,0.0,3.0


## Rows per state

In [6]:
states = df["State"].value_counts()
print(f"{states.size} states")
states.head(15).to_frame("rows").assign(share_pct=lambda t: (t.rows / len(df) * 100).round(2))

49 states


,rows,share_pct
State,,
CA,113274,22.65
FL,56710,11.34
TX,37355,7.47
SC,24737,4.95
NY,22594,4.52
NC,21750,4.35
VA,19515,3.90
PA,19351,3.87
MN,12333,2.47


## Save

In [7]:
df.to_parquet(DATA / "sample.parquet", index=False)
print(f"CSV {RAW.stat().st_size / 1e6:.0f} MB -> Parquet {(DATA / 'sample.parquet').stat().st_size / 1e6:.0f} MB")

CSV 197 MB -> Parquet 64 MB


## Recorded class counts (quote in the report)

| Severity | Sample (500,000) | Sample % | Full data (7,728,394) | Full % |
|---|---|---|---|---|
| 1 | 4,274 | 0.85 | 67,366 | 0.87 |
| 2 | 398,142 | 79.63 | 6,156,981 | 79.67 |
| 3 | 84,520 | 16.90 | 1,299,337 | 16.81 |
| 4 | 13,064 | 2.61 | 204,710 | 2.65 |

**Representative:** every class share is within 0.1 pp of the full data; all 49 states and 2016–2023 are covered (2023 = Jan–Mar only).

**Drift warning for Phase 7:** the Severity mix changes strongly by year. Severity 3 falls from ~32% (2017–18) to 0% (2023), and Severity 1 appears almost only in 2020 and 2022. A time-based split will score lower than a random split.